# 20260716_EDA_2023_전국_시도분리정제

- 작성자: 이정연
- #7 이슈 참고 

In [1]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

# 연도 및 경로 설정
YEAR = 2023

DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")


def get_sido_dir(sido: str) -> Path:
    """현재 폴더 구조(data/interim/{지역})를 유지하며 지역 경로를 한 곳에서 생성한다."""
    path = INTERIM_DIR / sido
    path.mkdir(parents=True, exist_ok=True)
    return path


# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

In [2]:
# 데이터 로드
file_2023 = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2023년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획)_칼럼정렬.xlsx"
)
df_raw = pd.read_excel(file_2023, sheet_name="정리본_자동")

print(df_raw.shape)
df_raw.head(10)

(7712, 12)


,지역,세부사업명,사업분류재정구분,2023년 예산,2022년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,·,7345192,7255267,89925,1,·,1,3,5,데이터행
1,서울,1. 함께 일하고 함께 돌보는 사회(공통),·,2845476,2862305,-16829,-1,·,1,3,6,데이터행
2,서울,저출생 인식개선\n캠페인,공통,34,54,-20,-37,"ㅇ지원대상: 서울시민\nㅇ지원내용: 서울100인의 아빠단, 저출산극복, 인식 개선 캠페인 등",1,3,7,데이터행
3,서울,입양아동 가족지원,공통,4443,4763,-320,-7,"ㅇ지원대상: 입양특례법에 의한 18세미만 입양아동 및 그 가정\nㅇ지원내용: 입양아동양육보조금, 심리치료비등",1,3,8,데이터행
4,서울,국공립어린이집 등 보육서비스 수준제고,공통,341815,330578,11237,3,ㅇ지원대상: 국공립 어린이집 등 ㅇ지원내용: 보육교직원 인건비 지원,1,3,9,데이터행
5,서울,어린이집 이용 영유아 무상보육 확대,공통,730323,805544,-75221,-9,ㅇ지원대상: 만0~2세 영아\nㅇ지원내용: 어린이집이용영아대상바우처지원,1,3,10,데이터행
6,서울,초등돌봄 공적 인프라 확충,공통,46207,58731,-12524,-21,ㅇ지원대상: 만6세~12세 아동(소득 무관)\nㅇ지원내용: 돌봄서비스제공,1,3,11,데이터행
7,서울,육아종합지원센터 운영,공통,2476,1963,513,26,"ㅇ지원대상: 어린이집 및 영유아 양육 가정\nㅇ지원내용\n- 어린이집 지원사업: 표준보육과정 교육, 대체교사 지원, 어린이집 맞춤 컨설팅, 장애아지원 교육 보육교직원 교육 및 상담, 서울형 모아어린이집 운영지원 등\n- 가정양육 지원사업: 장난감도서관 운영, 시간제 보육 관리, 우리동네 보육반장 운영지원 등\n- 자치구육아종합지원센터 운영지원 등",1,3,12,데이터행
8,서울,가정양육수당 지원,공통,39943,109686,-69743,-64,ㅇ지원대상: 시설 미이용 86개월 미만 아동\nㅇ지원내용: 15~20만원 차등 지원,1,3,13,데이터행
9,서울,엄마아빠 양육비 지원(부모급여),공통,345120,72832,272288,374,"ㅇ지원대상: 만 0~1세 아동('22년 이후 출생)\nㅇ지원내용: 만0세 70만원, 만1세 35만원 지원",1,3,14,데이터행


In [3]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (7712, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2023년 예산    object
2022년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.000
사업분류재정구분    0.000
2023년 예산    0.000
2022년 예산    0.000
증감액         0.000
비율          0.108
주요내용        0.014
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1313
경남     741
부산     732
충남     670
전남     638
강원     547
충북     494
전북     413
인천     346
울산     284
대구     276
경북     274
대전     245
서울     240
광주     207
제주     170
세종     122
Name: count, dtype: int64


In [4]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 547
경기 1313
경남 741
경북 274
광주 207
대구 276
대전 245
부산 732
서울 240
세종 122
울산 284
인천 346
전남 638
전북 413
제주 170
충남 670
충북 494


In [5]:
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)
df_raw["2023년 예산"] = pd.to_numeric(
    df_raw["2023년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
)

In [6]:
df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계
지역,,,
강원,2,537,8
경기,2,1303,8
경남,2,731,8
경북,2,264,8
광주,2,198,7
대구,2,266,8
대전,2,235,8
부산,2,722,8
서울,2,230,8


In [7]:
# 광주 중분류_소계 행 확인
df_gwangju = sido_dfs["광주"]
df_gwangju["사업행구분"] = df_gwangju["세부사업명"].apply(classify_row)

print("[광주] 중분류_소계 행 확인")
print("==================================================")
print(df_gwangju[df_gwangju["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

print("\n")

# 비교용으로 서울 출력
df_seoul = sido_dfs["서울"]
df_seoul["사업행구분"] = df_seoul["세부사업명"].apply(classify_row)

print("[서울] 중분류_소계 행 확인")
print("==================================================")
print(df_seoul[df_seoul["사업행구분"] == "중분류_소계"]["세부사업명"])
print("==================================================")

[광주] 중분류_소계 행 확인
1595      1. 함께 일하고 함께 돌보는 사회(공통)
1608        2. 건강하고 능동적인 고령사회(공통)
1619    3. 모두의 역량이 고루 발휘되는 사회(공통)
1627      1. 함께 일하고 함께 돌보는 사회(자체)
1707     2. 건강하고 능동적인 고령사회 구축(자체)
1750    3. 모두의 역량이 고루 발휘되는 사회(자체)
1790         4.인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


[서울] 중분류_소계 행 확인
1        1. 함께 일하고 함께 돌보는 사회(공통)
35         2. 건강하고 능동적인 고령사회(공통)
44     3. 모두의 역량이 고루 발휘되는 사회(공통)
48         4. 인구구조 변화에 대한 적응(공통)
52       1. 함께 일하고 함께 돌보는 사회(자체)
122     2. 건강하고 능동적인 고령사회 구축(자체)
169    3. 모두의 역량이 고루 발휘되는 사회(자체)
214        4. 인구구조 변화에 대한 적응(자체)
Name: 세부사업명, dtype: str


-> 광주에는 `4. 인구구조 변화에 대한 적응(공통)`이 없음.

-> 원본에도 없는 것 확인. 

-> 일단 없는 채로 둠. (따로 0으로 reindex 하지 않음.)

In [8]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

(7543, 15)
['세부사업명', '사업분류재정구분', '2023년 예산', '2022년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2023년 예산,2022년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 인식개선\n캠페인,공통,34.0,54,-20,-37,"ㅇ지원대상: 서울시민\nㅇ지원내용: 서울100인의 아빠단, 저출산극복, 인식 개선 캠페인 등",1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,입양아동 가족지원,공통,4443.0,4763,-320,-7,"ㅇ지원대상: 입양특례법에 의한 18세미만 입양아동 및 그 가정\nㅇ지원내용: 입양아동양육보조금, 심리치료비등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,국공립어린이집 등 보육서비스 수준제고,공통,341815.0,330578,11237,3,ㅇ지원대상: 국공립 어린이집 등 ㅇ지원내용: 보육교직원 인건비 지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,어린이집 이용 영유아 무상보육 확대,공통,730323.0,805544,-75221,-9,ㅇ지원대상: 만0~2세 영아\nㅇ지원내용: 어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,초등돌봄 공적 인프라 확충,공통,46207.0,58731,-12524,-21,ㅇ지원대상: 만6세~12세 아동(소득 무관)\nㅇ지원내용: 돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [18]:
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()

original_unique_count = (
    df_leaf.groupby("지역")["세부사업명"]
    .nunique()
    .sort_values(ascending=False)
    .rename("원본_세부사업명_고유개수")
)

print("전체 원본 세부사업명 고유 개수: ", df_leaf["세부사업명"].nunique())

# 지역 및 공통/자체 별로 나눠보기
original_unique_count = (
    df_leaf.groupby(["지역", "사업분류재정구분"])["세부사업명"]
    .nunique()
    .rename("원본_세부사업명_고유개수")
    .unstack(fill_value=0)
)

original_unique_count["합계"] = original_unique_count.sum(axis=1)

display(original_unique_count.sort_values("합계", ascending=False))

전체 원본 세부사업명 고유 개수:  6753


사업분류재정구분,공통,자체,합계
지역,,,
경기,53,1190,1243
경남,93,637,730
부산,105,608,713
충남,193,416,609
전남,79,517,596
강원,117,401,518
충북,50,425,475
전북,77,316,393
인천,95,241,336


In [19]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[-]")
BULLET_PATTERN = re.compile(r"(?:(?<=^)|(?<=\s))[ㅇ○◦□▪·o\-]\s*")


def clean_text(series: pd.Series, strip_bullet: bool = False) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리하고 필요 시 불릿 기호를 제거"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        x = re.sub(r"\s+", " ", x).strip()
        if strip_bullet:
            x = BULLET_PATTERN.sub("", x)
        return x

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_bullet=True)
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

# 공통/자체 구분은 값 내부의 모든 공백까지 제거
df_leaf["사업분류재정구분"] = clean_text(df_leaf["사업분류재정구분"]).str.replace(
    r"\s+", "", regex=True
)

df_leaf.head()

,세부사업명,사업분류재정구분,2023년 예산,2022년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 인식개선 캠페인,공통,34.0,54,-20,-37,"지원대상: 서울시민 지원내용: 서울100인의 아빠단, 저출산극복, 인식 개선 캠페인 등",1,3,7,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
3,입양아동 가족지원,공통,4443.0,4763,-320,-7,"지원대상: 입양특례법에 의한 18세미만 입양아동 및 그 가정 지원내용: 입양아동양육보조금, 심리치료비등",1,3,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
4,국공립어린이집 등 보육서비스 수준제고,공통,341815.0,330578,11237,3,지원대상: 국공립 어린이집 등 지원내용: 보육교직원 인건비 지원,1,3,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
5,어린이집 이용 영유아 무상보육 확대,공통,730323.0,805544,-75221,-9,지원대상: 만0~2세 영아 지원내용: 어린이집이용영아대상바우처지원,1,3,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울
6,초등돌봄 공적 인프라 확충,공통,46207.0,58731,-12524,-21,지원대상: 만6세~12세 아동(소득 무관) 지원내용: 돌봄서비스제공,1,3,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울


In [20]:
# 예산 컬럼에서 숫자로 변환이 안 되는 이상값 확인하기
mask_non_numeric = (
    pd.to_numeric(df_leaf["2023년 예산"], errors="coerce").isna() & df_leaf["2023년 예산"].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", "2023년 예산"]])

,지역,세부사업명,2023년 예산


-> 2023년 예산 값이 숫자로 변환 x

-> 비에산이 어떤의미인지 파악 필요

-> 일반적으로 비에산의 뜻

- 기존 인력/조직으로 처리 (외부 용역이나 구매 없이 담당 부서 자체 업무로 수행)
- 다른 사업 예산에 이미 포함되어 있어 별도 계산하지 않음. 
- 협약 / 홍보 / 캠페인성 사업처럼 실질적 지출이 발생하지 않는 활동 

-> 아마 여기서 1번에 해당하는 사례일듯. (전남 - 지능형 CCTV 실종자(치매,미아)찾기)

In [21]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, 예산 없음 표기(- · 비예산)는 0으로 처리 후 숫자로 변환"""
    zero_budget_tokens = ["-", "·", "비예산"]
    cleaned = series.astype(str).str.replace(",", "")
    cleaned = cleaned.replace(zero_budget_tokens, 0)
    return pd.to_numeric(cleaned, errors="coerce")


# 이전 셀 실행 여부와 무관하게 독립적으로 동작하도록 df_raw에서 다시 변환
df_raw["2023년_예산_num"] = to_numeric_budget(df_raw["2023년 예산"])
df_raw["2022년_예산_num"] = to_numeric_budget(df_raw["2022년 예산"])
df_raw["증감액_num"] = to_numeric_budget(df_raw["증감액"])
df_raw["증감액_재계산"] = df_raw["2023년_예산_num"] - df_raw["2022년_예산_num"]

# 실제로 예산이 감소한 행(증감액_재계산 < 0)만 골라서, 증감액 컬럼 부호를 대조
감소행 = df_raw[df_raw["증감액_재계산"] < -0.5].copy()
감소행["부호_소실"] = 감소행["증감액_num"] > 0.5

print("예산 감소 행 수:", len(감소행))
print("그중 증감액이 양수로 찍힌(부호 소실) 행 수:", 감소행["부호_소실"].sum())

# df_leaf는 df_raw와 분리된 복사본이므로 동일 규칙으로 별도 계산
df_leaf["2023년_예산_num"] = to_numeric_budget(df_leaf["2023년 예산"])
df_leaf["2022년_예산_num"] = to_numeric_budget(df_leaf["2022년 예산"])
df_leaf["증감액_재계산"] = df_leaf["2023년_예산_num"] - df_leaf["2022년_예산_num"]
df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"] / df_leaf["2022년_예산_num"].replace(0, np.nan) * 100
).round(1)

df_leaf[
    ["세부사업명", "2023년_예산_num", "2022년_예산_num", "증감액_재계산", "증감율_재계산"]
].head(10)

예산 감소 행 수: 1603
그중 증감액이 양수로 찍힌(부호 소실) 행 수: 40


,세부사업명,2023년_예산_num,2022년_예산_num,증감액_재계산,증감율_재계산
2,저출생 인식개선 캠페인,34.0,54,-20.0,-37.0
3,입양아동 가족지원,4443.0,4763,-320.0,-6.7
4,국공립어린이집 등 보육서비스 수준제고,341815.0,330578,11237.0,3.4
5,어린이집 이용 영유아 무상보육 확대,730323.0,805544,-75221.0,-9.3
6,초등돌봄 공적 인프라 확충,46207.0,58731,-12524.0,-21.3
7,육아종합지원센터 운영,2476.0,1963,513.0,26.1
8,가정양육수당 지원,39943.0,109686,-69743.0,-63.6
9,엄마아빠 양육비 지원(부모급여),345120.0,72832,272288.0,373.9
10,부모 모니터링단 운영,206.0,243,-37.0,-15.2
11,공동육아나눔터,2069.0,2030,39.0,1.9


In [22]:
# 지역별로 부호 소실 비율 확인
지역별_부호소실_비율 = (
    감소행.groupby("지역")["부호_소실"].mean().mul(100).round(1).sort_values(ascending=False)
)
지역별_부호소실_비율

지역
경기    13.4
서울     6.0
세종     3.4
충남     1.3
제주     0.0
전북     0.0
전남     0.0
인천     0.0
울산     0.0
강원     0.0
부산     0.0
대전     0.0
대구     0.0
광주     0.0
경북     0.0
경남     0.0
충북     0.0
Name: 부호_소실, dtype: float64

In [23]:
# 내부 공백만 다른 세부사업명 확인 - 내부 공백을 없앴을 때 세부사업명이 같은 경우
df_leaf["세부사업명_공백제거"] = (
    df_leaf["세부사업명"].astype("string").str.replace(r"\s+", "", regex=True)
)

space_name_summary = (
    df_leaf.groupby(["지역", "사업분류재정구분", "세부사업명_공백제거"], dropna=False)
    .agg(
        원본사업명수=("세부사업명", "nunique"),
        원본사업명=("세부사업명", lambda x: " | ".join(pd.unique(x.dropna().astype(str)))),
        행수=("세부사업명", "size"),
        당해예산종류수=("2023년_예산_num", "nunique"),
        당해예산목록=("2023년_예산_num", lambda x: list(x.dropna())),
        전년도예산종류수=("2022년_예산_num", "nunique"),
        전년도예산목록=("2022년_예산_num", lambda x: list(x.dropna())),
    )
    .reset_index()
)

# 공백을 제거했을 때는 같지만, 원래 사업명 표기가 2개 이상인 경우
space_name_qa = space_name_summary[space_name_summary["원본사업명수"] > 1].copy()

# 예산도 서로 다른지 표시
space_name_qa["당해예산차이"] = space_name_qa["당해예산종류수"] > 1
space_name_qa["전년도예산차이"] = space_name_qa["전년도예산종류수"] > 1

print(f"[{YEAR}년 예산 차이]")
print(space_name_qa["당해예산차이"].value_counts())
print("=====================================")
print(f"[{YEAR - 1}년 예산 차이]")
print(space_name_qa["전년도예산차이"].value_counts())
print("=====================================")
print("[차이조합]")
print(space_name_qa.groupby(["당해예산차이", "전년도예산차이"]).size())

[2023년 예산 차이]
당해예산차이
True    79
Name: count, dtype: int64
[2022년 예산 차이]
전년도예산차이
True    79
Name: count, dtype: int64
[차이조합]
당해예산차이  전년도예산차이
True    True       79
dtype: int64


-> 같은 지역, 같은 공통/자체 구분, 세부사업명에서 모든 공백을 제거했을 때 같은 이름이 되는 그룹 = 79개 

-> 79개 그룹 모두에서:
  - 2023년 예산이 서로 다름.
  - 2022년 예산도 서로 다름. 


-> 따라서 내부 공백을 모두 제거하고 자동 합산하는 방식은 위험함.

In [24]:
space_name_detail = df_leaf.merge(
    space_name_qa[
        ["지역", "사업분류재정구분", "세부사업명_공백제거", "당해예산차이", "전년도예산차이"]
    ],
    on=["지역", "사업분류재정구분", "세부사업명_공백제거"],
    how="inner",
    validate="many_to_one",
)

display(
    space_name_detail[
        [
            "지역",
            "사업분류재정구분",
            "세부사업명_공백제거",
            f"{YEAR}년_예산_num",
            f"{YEAR - 1}년_예산_num",
            "주요내용",
            "원본행",
            "당해예산차이",
            "전년도예산차이",
        ]
    ]
    .sort_values(
        [
            "당해예산차이",
            "전년도예산차이",
            "지역",
            "사업분류재정구분",
            "세부사업명_공백제거",
            "원본행",
        ],
        ascending=[False, False, True, True, True, True],
    )
    .head(50)
)

,지역,사업분류재정구분,세부사업명_공백제거,2023년_예산_num,2022년_예산_num,주요내용,원본행,당해예산차이,전년도예산차이
74,강원,자체,노인무료급식지원,850.0,1000,가정형편 등으로 결식의 우려가 있는 저소득 노인 들에게 무상급식을 제공하여 취약계층 노인의 건강 증진 도모,4698,True,True
75,강원,자체,노인무료급식지원,343.0,284,결식우려 노인 무료급식 지원,4705,True,True
70,강원,자체,연중아동급식지원,102.0,116,결식우려 아동들이 건전하게 자랄 수 있도록 급식 지원을 통해 보편적 생활지원,4426,True,True
71,강원,자체,연중아동급식지원,198.0,155,지역아동센터 3개소 이용아동에게 석식제공,4504,True,True
72,강원,자체,출산장려금지원사업,438.0,375,출산 장려금 지원,4598,True,True
73,강원,자체,출산장려금지원사업,260.0,260,"출산, 양육 경제적부담 완화(첫째 50만원, 둘째 70 만원, 셋째 100만원, 넷째이상 200만원 )",4605,True,True
47,경기,자체,경력단절여성취업지원,158.0,158,경력단절 여성의 노동시장 재진입 지원강화,3746,True,True
64,경기,자체,경력단절여성취업지원,44.0,72,경력단절여성의 직업교육훈련 과정 운영,3998,True,True
16,경기,자체,난임부부시술비지원사업,32075.0,29722,"아이를 원하는 난임부부의 시술비 지원(건강보험 적용시술- 인공5회,신선7회, 동결5회)",2900,True,True
28,경기,자체,난임부부시술비지원사업,210.0,189,"사업대상: 부부 모두 신청일 기준 최근 6개월 이상 광주시에 거주자로 중위소득 180% 초과 난임부부 지원내용: 난임시술비 중 일부 전액본인부담금 90%, 비급여 지원금액 상한범위 내 지원",3254,True,True


In [28]:
print("[지역별 공백 표기가 다른 동명 사업군 수]")

space_name_count_by_region = (
    space_name_qa.groupby(
        [
            "지역",
            "사업분류재정구분",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

space_name_count_by_region["합계"] = space_name_count_by_region.sum(axis=1)


display(space_name_count_by_region.sort_values("합계", ascending=False))

[지역별 공백 표기가 다른 동명 사업군 수]


사업분류재정구분,공통,자체,합계
지역,,,
경기,0,22,22
충남,15,7,22
전남,1,10,11
충북,0,11,11
부산,0,7,7
강원,0,3,3
전북,0,3,3


In [ ]:
#

In [ ]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
df_leaf["2023년 예산_num"] = to_numeric_budget(df_leaf["2023년 예산"])

leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])["2023년 예산_num"]
    .sum()
    .reset_index()
    .rename(columns={"2023년 예산_num": "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", "2023년 예산"]
].rename(columns={"세부사업명": "중분류", "2023년 예산": "원본_소계값"})
subtotal["원본_소계값"] = to_numeric_budget(subtotal["원본_소계값"])

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["결과"].value_counts()

In [ ]:
qa["차이"] = qa["leaf_합계"] - qa["원본_소계값"]

# 결과 컬럼을 항상 채워서, 저장 파일이 비어있어도 "검증은 했다"는 게 남도록 함
qa["결과"] = qa["차이"].abs().le(0.01).map({True: "일치", False: "불일치"})

qa_mismatch = qa[qa["결과"] == "불일치"]
print(f"검증 대상: {len(qa)}건 / 불일치: {len(qa_mismatch)}건")

display(qa_mismatch[["지역", "대분류", "중분류", "원본_소계값", "leaf_합계", "차이"]])

86	충북	Ⅱ. 지자체 자체사업	4.인구구조 변화에 대한 적응(자체)	30807	30553	-254


-> 가장 큰 차이 

# QA 불일치 원인 확인 - 원본 Table 1 대조

## 충북
가장 차이가 큼. 

In [ ]:
df_chungbuk_raw = sido_dfs["충북"]

mask_nan = pd.to_numeric(
    df_chungbuk_raw["2023년 예산"].astype(str).str.replace(",", "").replace("-", 0), errors="coerce"
).isna()

display(df_chungbuk_raw.loc[mask_nan, ["세부사업명", "2023년 예산"]])

In [ ]:
df_chungbuk = df_labeled[df_labeled["지역"] == "충북"]
print(df_chungbuk["사업행구분"].value_counts())
display(df_chungbuk[df_chungbuk["사업행구분"] == "중분류_소계"]["세부사업명"])

In [ ]:
# 원본 Table 1 로드
file_2023_original = (
    DATA_DIR / "세부사업표추출_2023년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획).xlsx"
)
df_table1 = pd.read_excel(file_2023_original, sheet_name="Table 1", header=None)

page_header_gap_size = 3


def show_table1_around(center_excel_row: int, window: int, label: str):
    """Table 1 원본 시트에서 특정 엑셀 행(1-indexed) 주변을 보여줌"""
    start, end = center_excel_row - window, center_excel_row + window
    print(f"--- {label} (Table1 엑셀행 {start}~{end}) ---")
    # 1,3열은 병합셀로 생긴 빈 열이라 제외, 나머지는 알아보기 쉬운 이름으로 표시
    view = df_table1.iloc[start - 1 : end, [0, 2, 3]].dropna(axis=1, how="all")
    view.columns = ["세부사업명", "공통/자체", "예산"]
    display(view)

In [ ]:
# 충북 "4. 인구구조 변화 대한 적응(자체)" leaf 행에서 원본행이 연속되지 않는(결번) 구간 탐지
target_mask = df_labeled["중분류"].str.contains("인구구조", na=False) & df_labeled[
    "중분류"
].str.contains("자체", na=False)

df_chungbuk_cat4 = df_labeled[
    (df_labeled["지역"] == "충북") & target_mask & (df_labeled["사업행구분"] == "세부사업")
]

original_row_numbers = df_chungbuk_cat4["원본행"].tolist()
gap_ranges = [(a, b) for a, b in zip(original_row_numbers, original_row_numbers[1:]) if b - a > 1]
print("결번 구간: ", gap_ranges)

for gap_start, gap_end in gap_ranges:
    gap_size = gap_end - gap_start
    status = "정상 추정(페이지 헤더 반복)" if gap_size == page_header_gap_size else "확인 필요"
    show_table1_around(
        (gap_start + gap_end) // 2,
        window=gap_size,
        label=f"{gap_start}~{gap_end} 결번 구간 [{status}]",
    )

# 4. 인구구조 변화에 대한 적응(자체) 소계 행의 실제 원본행을 가져와 그 주변 확인
subtotal_row = df_labeled[
    (df_labeled["지역"] == "충북")
    & df_labeled["세부사업명"].str.contains("인구구조", na=False)
    & df_labeled["세부사업명"].str.contains("자체", na=False)
    & (df_labeled["사업행구분"] == "중분류_소계")
]

show_table1_around(
    int(subtotal_row["원본행"].iloc[0]), window=3, label="4.인구구조(자체) 소계 전후"
)

-> 원본 보고서 자체의 소계 오기재 

## 부산
작은 오차들 중 가장 큼 (=4)

In [ ]:
target_mask = df_labeled["중분류"].str.contains("함께 일하고", na=False) & df_labeled[
    "중분류"
].str.contains("자체", na=False)

df_busan_cat1 = df_labeled[
    (df_labeled["지역"] == "부산") & target_mask & (df_labeled["사업행구분"] == "세부사업")
]

original_row_numbers = df_busan_cat1["원본행"].tolist()
gap_ranges = [(a, b) for a, b in zip(original_row_numbers, original_row_numbers[1:]) if b - a > 1]
print("결번 구간: ", gap_ranges)

for gap_start, gap_end in gap_ranges:
    gap_size = gap_end - gap_start
    status = "정상 추정(페이지 헤더 반복)" if gap_size == page_header_gap_size else "확인 필요"
    show_table1_around(
        (gap_start + gap_end) // 2,
        window=gap_size,
        label=f"{gap_start}~{gap_end} 결번 구간 [{status}]",
    )

# 4. 인구구조 변화에 대한 적응(자체) 소계 행의 실제 원본행을 가져와 그 주변 확인
subtotal_row = df_labeled[
    (df_labeled["지역"] == "부산")
    & df_labeled["세부사업명"].str.contains("함께 일하고", na=False)
    & df_labeled["세부사업명"].str.contains("자체", na=False)
    & (df_labeled["사업행구분"] == "중분류_소계")
]

show_table1_around(
    int(subtotal_row["원본행"].iloc[0]), window=3, label="1.함께일하고(자체) 소계 전후"
)

In [ ]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            "2023년_예산_num": "당해예산",
            "2022년_예산_num": "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=2023)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

print(df_leaf_final.shape)
display(df_leaf_final.head())

In [ ]:
# 정제 완료된 실제 세부사업을 기준으로 지역 내 사업명 중복 확인
# 소계·반복 머리글은 이미 제외되었으며, 지역·공통/자체 구분·사업명이 모두 같은 경우만 중복으로 본다.
duplicate_business_names = (
    df_leaf_final.dropna(subset=["지역", "사업분류재정구분", "세부사업명"])
    .groupby(["지역", "사업분류재정구분", "세부사업명"], dropna=False)
    .size()
    .rename("중복건수")
    .reset_index()
    .query("중복건수 > 1")
    .sort_values(
        ["지역", "사업분류재정구분", "중복건수", "세부사업명"],
        ascending=[True, True, False, True],
    )
)

print("[지역별 중복 세부사업명 수]\n", duplicate_business_names.groupby("지역").size())
display(duplicate_business_names)

# 주요내용 구조 패턴 검사 
- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다. 

In [ ]:
def dedup_label(text: str) -> str:
    """지원대상 : 지원대상 : ...  처럼 같은 라벨이 연속으로 중복된 걸 하나로 정리"""
    if pd.isna(text):
        return text
    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )
    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)
    return text


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [ ]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

In [ ]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}
    m = support_pattern_broad.match(text)
    if m:
        return {"지원대상": m.group(2).strip(), "지원내용": m.group(4).strip()}
    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.json_normalize(df_leaf_final["주요내용"].apply(extract_via_regex))
df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

In [ ]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

# 세부사업명 오탈자 / 중복 탐지
- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다. 
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.

In [ ]:
from itertools import combinations
from rapidfuzz import fuzz

In [ ]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values("유사도", ascending=False)


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

In [ ]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

In [ ]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confidenet = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confidenet)

# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.

In [ ]:
import os
import json
import yaml
from openai import OpenAI
from tqdm import tqdm

tqdm.pandas()

with open("../configs/llm_extraction.yaml") as f:
    llm_cfg = yaml.safe_load(f)

client = OpenAI(
    api_key=os.environ["UPSTAGE_API_KEY"],
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def extract_numbers(text) -> list:
    """숫자(금액·비율·연령 등) 시퀀스만 뽑아 리스트로 반환 - 원문/정제문장 보존 검증용"""
    if pd.isna(text):
        return []
    return re.findall(r"\d+", text)


def numbers_preserved(original, cleaned) -> bool:
    """정제 과정에서 숫자가 그대로 보존됐는지 확인 (다르면 검토 대상)"""
    return extract_numbers(original) == extract_numbers(cleaned)

sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"

PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


# 체크포인트 파일을 직접 읽어서 판별한다(df_leaf_final 병합을 기다릴 필요 없음).
# 이렇게 해야 LLM 처리 셀보다 앞에 둘 수 있고, 한 번의 순차 실행(Run All)만으로
# 재실행 대상까지 자동으로 포함되어 처리된다 (CodeRabbit 리뷰 반영).
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    affected_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
    affected_idx = checkpoint_df.index[affected_mask]

    if len(affected_idx) > 0:
        checkpoint_df = checkpoint_df.drop(index=affected_idx)
        checkpoint_df.to_csv(CHECKPOINT_PATH)

    print(
        "재실행 대상(체크포인트에서 제거):",
        len(affected_idx),
        "건 -> 남은 행수:",
        len(checkpoint_df),
    )
else:
    print("체크포인트 파일 없음 - 전체 신규 실행")

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"

# 숫자보존 실패 = LLM이 원문에 없는 내용을 만들어냈을 가능성이 높은 행  -> 원문으로 되돌림
mismatch_idx = df_leaf_final.index[~df_leaf_final["숫자보존"]]
print("되돌릴 행 수: ", len(mismatch_idx), "\n")
# display(df_leaf_final.loc[mismatch_idx, ["지역", "세부사업명", "주요내용", "주요내용_정제"]])

fixed_values = df_leaf_final.loc[mismatch_idx, "주요내용"].copy()
df_leaf_final.loc[mismatch_idx, "주요내용_정제"] = fixed_values

stll_different = (
    df_leaf_final.loc[mismatch_idx, "주요내용_정제"] != df_leaf_final.loc[mismatch_idx, "주요내용"]
)
print("대입 후에도 값이 다른 행: ", stll_different.sum(), "\n")

df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

for idx in mismatch_idx[:3]:
    orig = df_leaf_final.loc[idx, "주요내용"]
    clean = df_leaf_final.loc[idx, "주요내용_정제"]
    print(idx)
    print("주요내용: ", repr(orig)[:120])
    print("주요내용_정제: ", repr(clean)[:120])
    print("두 값 같은지", orig == clean if not (pd.isna(orig) and pd.isna(clean)) else "둘다 NaN")
    print("\n")

print("\n재검증 후 숫자 불일치 건수:", (~df_leaf_final["숫자보존"]).sum())

# 체크포인트 파일에도 반영
checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
checkpoint_df.loc[mismatch_idx, "주요내용_정제"] = fixed_values.values
checkpoint_df.to_csv(CHECKPOINT_PATH)

# 저장 확인
reloaded = pd.read_csv(CHECKPOINT_PATH, index_col=0)
still_bad_on_disk = (
    reloaded.loc[mismatch_idx, "주요내용_정제"] != df_leaf_final.loc[mismatch_idx, "주요내용"]
)
print("체크포인트 파일 갱신 확인 - 여전히 잘못된 행: ", still_bad_on_disk.sum())

# 최종 스키마

In [ ]:
for sido in df_leaf_final["지역"].unique():
    sido_leaf = df_leaf_final[df_leaf_final["지역"] == sido]
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = get_sido_dir(sido)

    sido_leaf.to_csv(
        sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv", index=False, encoding="utf-8-sig"
    )
    sido_labeled.to_csv(
        sido_dir / f"{YEAR}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장z
qa.to_csv(REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

In [ ]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")


print("\n숫자보존 불일치 총 건수:", (~df_leaf_final["숫자보존"]).sum())

In [ ]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

In [ ]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = get_sido_dir(sido_name) / f"{YEAR}_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")

# 2024_정제 테이블에서의 2023년도 예산과 2023_정제 테이블에서의 예산 비교 
세부사업명 + 사업분류재정구분이 같은 경우, 최종예산과 계획예산이 얼마나 달라졌는지 / 주요내용이 얼마나 달라졌는지 확인 필요

In [ ]:
def load_long(sido: str, source_year: int) -> pd.DataFrame:
    return pd.read_csv(get_sido_dir(sido) / f"{source_year}_{sido}_세부사업_정제_long.csv")


def compare_target_year_all_categories(
    target_year: int, sources: list[int], sido_list: list[str] | None = None
) -> pd.DataFrame:
    if sido_list is None:
        sido_list = sorted(df_leaf_final["지역"].unique())

    frames = []
    for source_year in sources:
        col_name = f"{target_year}년예산_{source_year}년문서"
        rows = []
        for sido in sido_list:
            try:
                df_long = load_long(sido, source_year)
            except FileNotFoundError as e:
                print(f"[건너뜀] {sido}: ({source_year}년 문서): {e}")
                continue
            sub = df_long[df_long["연도"] == target_year][["지역", "세부사업명", "예산액"]].rename(
                columns={"예산액": col_name}
            )
            rows.append(sub)

        frames.append(
            pd.concat(rows, ignore_index=True)
            if rows
            else pd.DataFrame(columns=["지역", "세부사업명", col_name])
        )

    result = frames[0]
    for f in frames[1:]:
        result = result.merge(f, on=["지역", "세부사업명"], how="inner")
        budget_cols = [c for c in result.columns if c.startswith(f"{target_year}년예산_")]
        result["일치"] = result[budget_cols[0]] == result[budget_cols[1]]
    return result

In [ ]:
all_comparison = compare_target_year_all_categories(target_year=2023, sources=[2023, 2024])

print("전체 겹치는 세부사업 수: ", len(all_comparison))
print("일치:", all_comparison["일치"].sum(), "/ 불일치:", (~all_comparison["일치"]).sum())

print("\n불일치 상세")
display(all_comparison[~all_comparison["일치"]].groupby("지역").size().sort_values(ascending=False))

In [ ]:
all_comparison["예산변동액"] = (
    all_comparison["2023년예산_2024년문서"] - all_comparison["2023년예산_2023년문서"]
)

all_comparison["예산변동률(%)"] = (
    all_comparison["예산변동액"] / all_comparison["2023년예산_2023년문서"].replace(0, np.nan) * 100
)

display(all_comparison.sort_values("예산변동률(%)", key=abs, ascending=False).head(20))

## 세부사업명 + 주요내용 까지 같이 비교
- 세부사업명이 일치하는 항목끼리, 예산변동률뿐만아니라 주요내용 텍스트도 나란히 두고 내용이 달라졌는지 확인한다.

In [ ]:
def load_year_detail(sido: str, source_year: int, target_year: int) -> pd.DataFrame:
    """soure_year 문서의 long 파일에서 target_year에 해당하는 예산액과 주요내용을 뽑는다"""
    df_long = load_long(sido, source_year)

    result = df_long.loc[
        df_long["연도"] == target_year,
        [
            "지역",
            "사업분류재정구분",
            "세부사업명",
            "예산액",
            "주요내용",
        ],
    ].copy()

    return result


target_year, sources = 2023, [2023, 2024]
sido_list = sorted(df_leaf_final["지역"].unique())
group_keys = [
    "지역",
    "사업분류재정구분",
    "세부사업명",
]

frames = []

for source_year in sources:
    rows = []
    for sido in sido_list:
        try:
            sub = load_year_detail(sido, source_year, target_year)
        except FileNotFoundError:
            continue
        rows.append(sub)

    combined = (
        pd.concat(rows, ignore_index=True)
        if rows
        else pd.DataFrame(columns=["지역", "사업분류재정구분", "세부사업명", "예산액", "주요내용"])
    )
    # 같은 지역 공통, 자체, 세부사업명이 여러 행이면 문서별로 먼저 집계함.
    combined = combined.groupby(group_keys, as_index=False, dropna=False).agg(
        예산액=("예산액", lambda x: pd.to_numeric(x, errors="coerce").sum(min_count=1)),
        주요내용=("주요내용", lambda x: " | ".join(pd.unique(x.dropna().astype(str)))),
        구성항목수=("세부사업명", "size"),
    )

    # 문서 출처를 구분할 수 있도록 컬럼명 변경
    combined = combined.rename(
        columns={
            "예산액": f"{target_year}년예산_{source_year}년문서",
            "주요내용": f"주요내용_{source_year}년문서",
            "구성항목수": f"구성항목수_{source_year}년문서",
        }
    )

    frames.append(combined)

detail_comparison = frames[0].merge(frames[1], on=group_keys, how="inner", validate="one_to_one")

initial_budget_col = f"{target_year}년예산_{sources[0]}년문서"
final_budget_col = f"{target_year}년예산_{sources[1]}년문서"

# 지역·공통/자체·세부사업명이 모두 같은 행에서 본예산과 최종예산의 일치 여부를 계산한다.
# 한쪽이라도 결측이면 일치로 보지 않는다.
detail_comparison["본예산_최종예산_일치"] = np.isclose(
    pd.to_numeric(detail_comparison[initial_budget_col], errors="coerce"),
    pd.to_numeric(detail_comparison[final_budget_col], errors="coerce"),
    equal_nan=False,
)

detail_comparison["예산변동률(%)"] = (
    (detail_comparison[final_budget_col] - detail_comparison[initial_budget_col])
    / detail_comparison[initial_budget_col].replace(0, np.nan)
    * 100
)

# 주요 내용 문자열이 달라졌는지 표시
detail_comparison["주요내용_변경"] = (
    detail_comparison[f"주요내용_{sources[0]}년문서"]
    != detail_comparison[f"주요내용_{sources[1]}년문서"]
)
display_cols = [
    "지역",
    "사업분류재정구분",
    "세부사업명",
    f"구성항목수_{sources[0]}년문서",
    f"구성항목수_{sources[1]}년문서",
    initial_budget_col,
    final_budget_col,
    "본예산_최종예산_일치",
    "예산변동률(%)",
    f"주요내용_{sources[0]}년문서",
    f"주요내용_{sources[1]}년문서",
    "주요내용_변경",
]

display(
    detail_comparison.sort_values("예산변동률(%)", key=abs, ascending=False)[display_cols].head(20)
)

In [ ]:
print("세부사업명 + 사업재정분류가 겹치는 세부사업 수: ", len(detail_comparison))
print(
    "본예산_최종예산_일치:",
    detail_comparison["본예산_최종예산_일치"].sum(),
    "/ 본예산_최종예산_불일치:",
    (~detail_comparison["본예산_최종예산_일치"]).sum(),
)

print("\n불일치 상세")
display(
    detail_comparison[~detail_comparison["본예산_최종예산_일치"]]
    .groupby("지역")
    .size()
    .sort_values(ascending=False)
)

In [ ]:
# 퍼지 매칭 적용 - 문자열 유사도 + 숫자 변경 여부 함께 보기
# 기존 '주요내용_변경'은 한 글자만 달라도 True인 완전일치 기준이므로 비교용으로만 보존한다.
# 실제 후속 분석은 아래에서 만드는 '주요내용_실질변경'을 사용한다.
import re

In [ ]:
# 텍스트 정규화 - 공백·문장부호·불릿 등 편집 차이를 제거해
# 정책 내용이 같은데 표기만 다른 경우가 변경으로 과대 분류되는 것을 줄인다.
def normalize_for_comparison(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[·ㆍ•○ㅇ□▪,./:;()\-]", "", text)

    return text


text_2023 = detail_comparison["주요내용_2023년문서"]
text_2024 = detail_comparison["주요내용_2024년문서"]

# NaN과 빈 문자열을 모두 결측로 본다. 결측끼리의 비교를 '일치'로 해석하지 않는다.
missing_2023 = text_2023.isna() | text_2023.fillna("").str.strip().eq("")
missing_2024 = text_2024.isna() | text_2024.fillna("").str.strip().eq("")

detail_comparison["주요내용_자료상태"] = np.select(
    [missing_2023 & missing_2024, missing_2023 | missing_2024],
    ["양쪽 모두 결측", "한쪽만 결측"],
    default="비교 가능",
)

# fuzz.ratio는 정규화된 두 문장의 유사도를 0~100으로 반환한다.
# 한쪽이라도 결측이면 유사도를 계산하지 않고 NaN으로 남겨 '판단 불가'로 구분한다.
detail_comparison["주요내용_유사도"] = [
    fuzz.ratio(normalize_for_comparison(old), normalize_for_comparison(new))
    if not (old_missing or new_missing)
    else np.nan
    for old, new, old_missing, new_missing in zip(text_2023, text_2024, missing_2023, missing_2024)
]

# 유사도 구간은 제출용 판정 기준이 아니라 구간별 표본 검수를 위한 임시 격자이다.
# 첫 경계를 -1로 두어 유사도 0점도 누락 없이 첫 구간에 포함한다.
detail_comparison["주요내용_변경수준"] = pd.cut(
    detail_comparison["주요내용_유사도"],
    bins=[-1, 80, 95, 100],
    labels=["변경 가능성 높음", "검토 필요", "변경 가능성 낮음"],
)


detail_comparison.groupby(
    pd.cut(
        detail_comparison["주요내용_유사도"],
        bins=[-1, 80, 95, 100],
        # labels=["변경 가능성 높음", "검토 필요", "변경 가능성 낮음"],
    ),
    observed=True,
).size()

In [ ]:
# 유사도가 높아도 금액·인원·연령·기간이 달라지면 실질적 변경일 수 있어 숫자를 별도 비교한다.
# 숫자 표기 차이로 인한 오탐을 줄이기 위해 68,020과 68020을 같은 값으로 비교한다.
def extract_comparison_numbers(text):
    if pd.isna(text):
        return []
    return re.findall(r"\d+(?:\.\d+)?", str(text).replace(",", ""))


detail_comparison["주요내용_숫자변경"] = [
    extract_comparison_numbers(old) != extract_comparison_numbers(new)
    if not (old_missing or new_missing)
    else pd.NA
    for old, new, old_missing, new_missing in zip(text_2023, text_2024, missing_2023, missing_2024)
]

# 검토 격자: 단순 True/False 판정 대신 결측·숫자 변경·표현 변경을 구분해 인적 검수 우선순위를 제공한다.
detail_comparison["주요내용_검토구분"] = np.select(
    [
        detail_comparison["주요내용_자료상태"].ne("비교 가능"),
        detail_comparison["주요내용_유사도"].eq(100)
        & detail_comparison["주요내용_숫자변경"].eq(False),
        detail_comparison["주요내용_유사도"].ge(95)
        & detail_comparison["주요내용_숫자변경"].eq(True),
        detail_comparison["주요내용_유사도"].ge(95),
    ],
    [
        "판단 불가",
        "변경없음",
        "숫자 변경 검토",
        "표현 변경 가능성",
    ],
    default="내용 변경 검토",
)

# 95점은 임시 기준이며, 구간별 표본 검수 후 조정해야 한다.
# 표현만 다른 95점 이상은 비변경, 숫자 변경은 실질 변경으로 간주한다.
detail_comparison["주요내용_실질변경"] = np.select(
    [
        detail_comparison["주요내용_자료상태"].ne("비교 가능"),
        detail_comparison["주요내용_검토구분"].isin(["숫자 변경 검토", "내용 변경 검토"]),
    ],
    [np.nan, 1.0],
    default=0.0,
)

detail_comparison.groupby(
    pd.cut(
        detail_comparison["주요내용_유사도"],
        bins=[-1, 60, 80, 90, 95, 99, 100],
    ),
    observed=True,
).size()

## 시각화 
- 예산변동률 전체 분포를 보고, 주요내용_변경 여부에 따라 변동폭이 다르게 나타나는지 비교 
- 주요내용 변경 여부와 예산변동 사이에 실제로 관련성이 있는가를 확인

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Seaborn 스타일을 먼저 적용해야 한글 폰트 설정이 덮어쓰여지지 않음
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
FIG_SIZE = (18, 7)  # (가로, 세로), inch

PALETTE = sns.color_palette("colorblind")

In [ ]:
# 분석 모집단 제한:
# 1) 본예산이 0이어서 변동률의 분모가 0인 행
# 2) 주요내용이 한쪽 또는 양쪽 결측이어서 실질 변경을 판단할 수 없는 행
# 위 행은 0이나 '변경 없음'으로 대체하지 않고 후속 비교에서 제외한다.
plot_df = detail_comparison.dropna(subset=["예산변동률(%)", "주요내용_실질변경"]).copy()
plot_df["주요내용_실질변경"] = plot_df["주요내용_실질변경"].astype(bool)

# 극단값이 히스토그램을 압도하지 않도록 표시범위만 클리핑
clip_range = 200
n_clipped = (
    (plot_df["예산변동률(%)"] < -clip_range) | (plot_df["예산변동률(%)"] > clip_range)
).sum()

plot_df["예산변동률_표시용"] = plot_df["예산변동률(%)"].clip(-clip_range, clip_range)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)

median_val = plot_df["예산변동률(%)"].median()

# 전체 분포
axes[0].hist(plot_df["예산변동률_표시용"], bins=40, color=PALETTE[0], edgecolor="white", alpha=0.85)
axes[0].axvline(0, color="gray", linestyle="--", linewidth=1.5, label=f"중앙값 {median_val:.1f}%")
axes[0].set_title(
    f"세부사업 예산변동율 분포 (n={len(plot_df):,}, ±{clip_range}% 밖 {n_clipped}건 경계값 표시)"
)
axes[0].set_xlabel("예산변동률 (%, 최종예산 - 본예산)")
axes[0].set_ylabel("세부사업 수")
axes[0].legend()

# 주요내용 변경 여부별 비교
sns.boxenplot(
    data=plot_df,
    x="주요내용_실질변경",
    y="예산변동률_표시용",
    hue="주요내용_실질변경",
    order=[False, True],
    hue_order=[False, True],
    palette=PALETTE[:2],
    legend=False,
    ax=axes[1],
)
axes[1].set_title("주요내용 변경 여부별 예산변동률")
axes[1].set_xlabel("주요내용 문구 변경 여부")
axes[1].set_ylabel("예산변동률(%, 표시 범위 제한)")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["변경 없음", "변경됨"])

plt.tight_layout()

out_path = Path("results")
out_path.mkdir(exist_ok=True)
plt.savefig(
    out_path / f"{YEAR}_에산변동률_분포.png", dpi=150, bbox_inches="tight", facecolor="white"
)

plt.show()

print(
    plot_df.groupby("주요내용_실질변경")["예산변동률(%)"]
    .describe()[["count", "mean", "50%", "std"]]
    .round(1)
)

-> 주요내용 변경 집단은 변경 없음 집단보다 평균 예산 변동률과 변동성이 크게 나타났으나 두 집단의 중앙값은 모두 0.

-> 변경 집단에 큰 폭의 예산 조정 사례가 더 크게 포함되었을 가능성이 높음

-> 통계적 유의성과 실질적 효과를 판단하려면 변동 발생 비율, 절대 변동률 및 효과 크기에 대한 추가 검정 필요

-> 따라서 위의 결과만 보고서는 주요내용이 변경된 사업의 예산변동률이 통계적으로 더 크다고 확정하기 어려움

In [ ]:
zero_summary = (
    plot_df.assign(변동없음=plot_df["예산변동률(%)"].eq(0))
    .groupby("주요내용_실질변경")
    .agg(
        전체사업수=("예산변동률(%)", "size"),
        변동없음수=("변동없음", "sum"),
        변동없음비율=("변동없음", "mean"),
        최소값=("예산변동률(%)", "min"),
        중앙값=("예산변동률(%)", "median"),
        최대값=("예산변동률(%)", "max"),
    )
)

zero_summary["변동없음비율"] *= 100
zero_summary.round(1)

In [ ]:
direction = np.select(
    [
        plot_df["예산변동률(%)"] < 0,
        plot_df["예산변동률(%)"] > 0,
    ],
    ["감액", "증액"],
    default="변동없음",
)  # 예산변동 방향 분류

direction_table = (
    pd.crosstab(
        plot_df["주요내용_실질변경"].map(
            {
                False: "변경없음",
                True: "변경됨",
            }
        ),
        direction,
        normalize="index",
    )
    .mul(100)
    .round(1)
)

direction_table = direction_table.reindex(columns=["감액", "변동없음", "증액"]).rename_axis(
    index="주요내용 변경 여부", columns="집단 내 예산변동 비율(%)"
)

direction_table

In [ ]:
plot_df["예산변동_여부"] = np.where(plot_df["예산변동률(%)"] == 0, "변동없음", "변동있음")

contingency_table = pd.crosstab(plot_df["주요내용_실질변경"], plot_df["예산변동_여부"])

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(contingency_table)
print(f"카이제곱 통계량: {chi2:.2f}")
print(f"p-value: {p_value:.4f}")

->  주요내용 변경 여부와 예산변동 발생 여부 간에는 통계적으로 유의한 관련성이 확인됨. 